In [51]:
from api_setup import setup

client, model_id = setup()

In [52]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": [
            {"text": text}
        ]
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant", 
        "content": [
            {"text": text}
        ]
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences
        },
    }
    
    if system:
        params["system"] = [{"text": system}]
    
    response = client.converse(**params)
    
    return response["output"]["message"]["content"][0]["text"]

In [53]:
import json

def generate_dataset():
    prompt = """
    Generate 3 AWS-related tasks that require Python, JSON, or Regex solutions.
    
    Focus on tasks that can be solved by writing a single Python function, 
    a single JSON object, or tasks that do not require writing much code.
    
    Example output:
    [
        {
            "task": "Description of task"
            "format": format of the solution either Python, JSON, or Regex
        },
        ...additional
    ]
    
    Please generate 3 objects.
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [54]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [55]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    output = chat(messages, stop_sequences=["```"])
    return output

In [56]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [57]:
def grade_syntax(test_case, output):
    format = test_case["format"]
    if format == "Python":
        return validate_python(output)
    elif format == "JSON":
        return validate_json(output)
    else:
        return validate_regex(output)

In [58]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [59]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(test_case, output)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": model_grade["reasoning"]
    }

In [60]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
                             
    return results



In [61]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

Average score: 7.916666666666667
[
  {
    "output": "\n{\n  \"regex_pattern\": \"^(?P<timestamp>\\\\d{4}-\\\\d{2}-\\\\d{2}T\\\\d{2}:\\\\d{2}:\\\\d{2}\\\\.\\\\d{3}Z)\\\\s+\\\\[(?P<log_level>\\\\w+)\\\\]\\\\s+(?P<error_message>.+)$\",\n  \"example_log_entry\": \"2024-01-15T10:30:45.123Z [ERROR] Connection timeout while connecting to database\",\n  \"groups\": {\n    \"timestamp\": \"2024-01-15T10:30:45.123Z\",\n    \"log_level\": \"ERROR\",\n    \"error_message\": \"Connection timeout while connecting to database\"\n  }\n}\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and error message using regex patterns",
      "format": "Regex"
    },
    "syntax_score": 10,
    "score": 8.0,
    "reasoning": "The solution demonstrates competent regex fundamentals with proper named groups and correct timestamp formatting. However, it lacks defensive programming for real-world CloudWatch variability. CloudWatch logs often have inconsiste